<a href="https://colab.research.google.com/github/e11106013/KG/blob/main/zh_to_nan_translation_ok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 安裝必要套件
%%capture
!pip install pandas datasets evaluate transformers[sentencepiece] accelerate sacrebleu


In [ ]:
#@title 資料集處理
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# 讀取 Google Sheet
url = "https://docs.google.com/spreadsheets/d/1-eIOxH8s3EAx3LuKC3iDyYxGGwhjlXpTspmZrgyWeoI/export?format=xlsx"
df = pd.read_excel(url)

# 標準化欄位名稱
df = df.rename(columns={"華語": "source", "台語": "target"})
df = df.dropna(subset=["source", "target"]).reset_index(drop=True)

# 打亂整個資料集
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 先對 target 唯一值做切分，避免 train/val target 重複
unique_targets = df.drop_duplicates(subset=["target"])
train_unique, val_unique = train_test_split(unique_targets, test_size=0.1, random_state=42)

# 取出驗證集的 index
val_idx = val_unique.index

# 將驗證集以外的資料放入訓練集
train_rest = df.drop(val_idx)

# 合併訓練集（包含唯一target和剩餘資料）
train_df = pd.concat([train_unique, train_rest]).drop_duplicates().reset_index(drop=True)
val_df = val_unique.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")

Train size: 114407
Validation size: 8051


In [ ]:
#@title 確認train/val target不重複
overlap_targets = set(train_df["target"]) & set(val_df["target"])
print("Train/Val 重複 target 數量:", len(overlap_targets))

# 轉成 Hugging Face DatasetDict
split_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})


Train/Val 重複 target 數量: 1998


In [ ]:
#@title 資料集轉成 Hugging Face DatasetDict
split_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
})


In [ ]:
#@title 模型與 tokenizer 載入設定
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
model_checkpoint = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

max_length = 128

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# 預處理函式：加翻譯前綴並做 tokenizer
def preprocess_function(examples):
    inputs = ["translate zh to nan: " + text for text in examples["source"]]
    targets = examples["target"]
    return tokenizer(inputs, text_target=targets, max_length=max_length, truncation=True)

In [ ]:
# Tokenize 資料
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names,
)


Map:   0%|          | 0/114407 [00:00<?, ? examples/s]

Map:   0%|          | 0/8051 [00:00<?, ? examples/s]

In [ ]:
#@title BLEU 評估函式設定
import evaluate
import numpy as np
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
#title  訓練參數設定,建立 Trainer 並啟動訓練
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
args = Seq2SeqTrainingArguments(
    output_dir="./mt5-zh-to-nan",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    seed=42,
    predict_with_generate=True,
    do_eval=True,
    optim="adamw_torch",             # Adam 優化器（Huggingface 目前只支援adamw, adamw_torch, adafactor等）
    lr_scheduler_type="linear",          # 線性學習率調度器
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

/tmp/ipython-input-2806258615.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e11106013 (e11106013-ttu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,9.311700
